# Analyzing the Largest Alzheimer EEG Dataset

This notebook explores the [Largest Alzheimer EEG Dataset](https://www.kaggle.com/datasets/codingyodha/largest-alzheimer-eeg-dataset) from Kaggle.

**Dataset**: Largest Alzheimer EEG Dataset  
**Source**: Kaggle  
**Purpose**: Explore and understand the structure of the EEG data for Alzheimer's detection

## Setup Instructions for Google Colab

1. Upload your Kaggle API credentials:
   - Go to your Kaggle account settings
   - Scroll down to "API" section
   - Click "Create New Token" to download `kaggle.json`
   - Upload `kaggle.json` to Colab when prompted in the first cell

2. Run all cells sequentially

In [ ]:
# Install required packages
!pip install kaggle pandas numpy matplotlib seaborn scipy -q

In [ ]:
# Setup Kaggle API
# Upload your kaggle.json file when prompted
from google.colab import files
import os
import json

# Upload kaggle.json file
uploaded = files.upload()

# Move kaggle.json to the correct location
for fn in uploaded.keys():
    if fn == 'kaggle.json':
        os.makedirs('/root/.kaggle', exist_ok=True)
        os.rename(fn, '/root/.kaggle/kaggle.json')
        os.chmod('/root/.kaggle/kaggle.json', 600)
        print("Kaggle API credentials configured successfully!")

In [ ]:
# Download the dataset
import kaggle

dataset_name = "codingyodha/largest-alzheimer-eeg-dataset"
output_dir = "./alzheimer_eeg_data"

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# Download dataset
print(f"Downloading dataset: {dataset_name}")
kaggle.api.dataset_download_files(dataset_name, path=output_dir, unzip=True)
print("Dataset downloaded successfully!")

# List downloaded files
import glob
files = glob.glob(f"{output_dir}/**/*", recursive=True)
print("\nDownloaded files:")
for f in sorted(files):
    if os.path.isfile(f):
        size = os.path.getsize(f) / (1024 * 1024)  # Size in MB
        print(f"  {f} ({size:.2f} MB)")

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("Libraries imported successfully!")

## Dataset Structure Exploration

Let's first understand what files are in the dataset and their structure.

In [ ]:
# Explore directory structure
data_path = Path(output_dir)

def explore_directory(path, level=0):
    """Recursively explore directory structure"""
    indent = "  " * level
    if path.is_dir():
        print(f"{indent}📁 {path.name}/")
        items = sorted(path.iterdir())
        for item in items:
            explore_directory(item, level + 1)
    else:
        size = path.stat().st_size / (1024 * 1024)  # Size in MB
        print(f"{indent}📄 {path.name} ({size:.2f} MB)")

print("Dataset directory structure:")
explore_directory(data_path)

In [ ]:
# Find all CSV files
csv_files = list(data_path.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV file(s):\n")
for csv_file in csv_files:
    print(f"  - {csv_file.relative_to(data_path)}")

# Find all other data files (e.g., .txt, .mat, .npy, etc.)
other_files = []
for ext in ['.txt', '.mat', '.npy', '.npz', '.h5', '.hdf5', '.edf', '.fif']:
    other_files.extend(list(data_path.rglob(f"*{ext}")))

if other_files:
    print(f"\nFound {len(other_files)} other data file(s):\n")
    for f in other_files:
        print(f"  - {f.relative_to(data_path)}")

## Load and Explore Data

Let's load the data files and examine their structure.

In [ ]:
# Load CSV files
dataframes = {}
for csv_file in csv_files:
    try:
        df_name = csv_file.stem
        print(f"\nLoading {csv_file.name}...")
        df = pd.read_csv(csv_file)
        dataframes[df_name] = df
        print(f"  ✓ Loaded successfully")
        print(f"  Shape: {df.shape} (rows, columns)")
        print(f"  Columns: {list(df.columns[:10])}{'...' if len(df.columns) > 10 else ''}")
    except Exception as e:
        print(f"  ✗ Error loading {csv_file.name}: {e}")

print(f"\n\nTotal dataframes loaded: {len(dataframes)}")

In [ ]:
# Display basic information about each dataframe
for name, df in dataframes.items():
    print(f"\n{'='*60}")
    print(f"DataFrame: {name}")
    print(f"{'='*60}")
    print(f"\nShape: {df.shape}")
    print(f"\nColumn names:")
    print(df.columns.tolist())
    print(f"\nFirst few rows:")
    print(df.head())
    print(f"\nData types:")
    print(df.dtypes)
    print(f"\nMissing values:")
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print("No missing values!")
    print(f"\nBasic statistics:")
    print(df.describe())

## Data Visualization

Let's create some visualizations to better understand the data distribution and characteristics.

In [ ]:
# Check for target/label columns (common names for classification)
target_keywords = ['label', 'target', 'class', 'diagnosis', 'alzheimer', 'disease', 'status', 'category']

for name, df in dataframes.items():
    print(f"\n{'='*60}")
    print(f"Analyzing: {name}")
    print(f"{'='*60}")
    
    # Find potential target columns
    potential_targets = [col for col in df.columns 
                        if any(keyword in col.lower() for keyword in target_keywords)]
    
    if potential_targets:
        print(f"\nPotential target/label columns: {potential_targets}")
        for target_col in potential_targets:
            if df[target_col].dtype == 'object' or df[target_col].nunique() < 20:
                print(f"\n  Distribution of '{target_col}':")
                print(df[target_col].value_counts())
    else:
        print("\nNo obvious target column found. Checking for numeric columns...")
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols[:10]}{'...' if len(numeric_cols) > 10 else ''}")

In [ ]:
# Create visualizations for each dataframe
for name, df in dataframes.items():
    print(f"\n{'='*60}")
    print(f"Visualizations for: {name}")
    print(f"{'='*60}")
    
    # Find target column
    target_col = None
    for col in df.columns:
        if any(keyword in col.lower() for keyword in target_keywords):
            if df[col].dtype == 'object' or df[col].nunique() < 20:
                target_col = col
                break
    
    # Plot 1: Target distribution (if found)
    if target_col:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Bar plot
        value_counts = df[target_col].value_counts()
        axes[0].bar(value_counts.index.astype(str), value_counts.values)
        axes[0].set_title(f'Distribution of {target_col}')
        axes[0].set_xlabel(target_col)
        axes[0].set_ylabel('Count')
        axes[0].tick_params(axis='x', rotation=45)
        
        # Pie chart
        axes[1].pie(value_counts.values, labels=value_counts.index.astype(str), autopct='%1.1f%%')
        axes[1].set_title(f'Proportion of {target_col}')
        
        plt.tight_layout()
        plt.show()
    
    # Plot 2: Distribution of numeric columns (sample if too many)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        n_cols_to_plot = min(6, len(numeric_cols))
        cols_to_plot = numeric_cols[:n_cols_to_plot]
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        axes = axes.flatten()
        
        for idx, col in enumerate(cols_to_plot):
            if idx < len(axes):
                axes[idx].hist(df[col].dropna(), bins=50, edgecolor='black', alpha=0.7)
                axes[idx].set_title(f'Distribution of {col}')
                axes[idx].set_xlabel(col)
                axes[idx].set_ylabel('Frequency')
        
        # Hide unused subplots
        for idx in range(len(cols_to_plot), len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        if len(numeric_cols) > n_cols_to_plot:
            print(f"\nNote: Only showing first {n_cols_to_plot} of {len(numeric_cols)} numeric columns")

In [ ]:
# Correlation heatmap for numeric columns (if dataset is not too large)
for name, df in dataframes.items():
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if len(numeric_cols) > 1 and len(numeric_cols) <= 50:  # Only for reasonable number of columns
        print(f"\n{'='*60}")
        print(f"Correlation Matrix for: {name}")
        print(f"{'='*60}")
        
        # Calculate correlation
        corr_matrix = df[numeric_cols].corr()
        
        # Plot heatmap
        plt.figure(figsize=(12, 10))
        sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, 
                   square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
        plt.title(f'Correlation Matrix - {name}')
        plt.tight_layout()
        plt.show()
    elif len(numeric_cols) > 50:
        print(f"\nSkipping correlation matrix for {name}: too many columns ({len(numeric_cols)})")

## Summary Statistics

Let's get a comprehensive summary of the dataset.

In [ ]:
# Summary statistics for all dataframes
summary_stats = []

for name, df in dataframes.items():
    stats = {
        'Dataset': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Numeric Columns': len(df.select_dtypes(include=[np.number]).columns),
        'Categorical Columns': len(df.select_dtypes(include=['object']).columns),
        'Missing Values': df.isnull().sum().sum(),
        'Memory Usage (MB)': df.memory_usage(deep=True).sum() / (1024 * 1024)
    }
    
    # Add target column info if found
    target_col = None
    for col in df.columns:
        if any(keyword in col.lower() for keyword in target_keywords):
            if df[col].dtype == 'object' or df[col].nunique() < 20:
                target_col = col
                break
    
    if target_col:
        stats['Target Column'] = target_col
        stats['Classes'] = df[target_col].nunique()
        stats['Class Distribution'] = str(dict(df[target_col].value_counts()))
    else:
        stats['Target Column'] = 'Not found'
        stats['Classes'] = 'N/A'
        stats['Class Distribution'] = 'N/A'
    
    summary_stats.append(stats)

# Create summary dataframe
summary_df = pd.DataFrame(summary_stats)
print("\n" + "="*80)
print("DATASET SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))

## Additional Exploration

Let's check for any time series patterns or EEG-specific characteristics.

In [ ]:
# Check for EEG channel names (common EEG electrode names)
eeg_channels = ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 
                'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz', 'A1', 'A2']

for name, df in dataframes.items():
    print(f"\n{'='*60}")
    print(f"EEG Channel Analysis: {name}")
    print(f"{'='*60}")
    
    # Check if column names match EEG channels
    found_channels = [col for col in df.columns if any(channel in str(col) for channel in eeg_channels)]
    
    if found_channels:
        print(f"\nFound {len(found_channels)} potential EEG channel columns:")
        print(found_channels[:20])  # Show first 20
        if len(found_channels) > 20:
            print(f"... and {len(found_channels) - 20} more")
    else:
        # Check for numeric columns that might be EEG channels
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        print(f"\nNo obvious EEG channel names found.")
        print(f"Found {len(numeric_cols)} numeric columns (might be EEG channels or features)")
        
        # Sample a few rows to see data patterns
        if len(numeric_cols) > 0:
            print(f"\nSample data from first numeric columns:")
            print(df[numeric_cols[:5]].head())

In [ ]:
# If we have time series data, plot sample signals
for name, df in dataframes.items():
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # If we have many numeric columns, they might be time points or features
    if len(numeric_cols) > 10:
        print(f"\n{'='*60}")
        print(f"Sample Signal Visualization: {name}")
        print(f"{'='*60}")
        
        # Plot first few rows as potential signals
        n_samples = min(3, df.shape[0])
        n_features = min(5, len(numeric_cols))
        
        fig, axes = plt.subplots(n_samples, 1, figsize=(15, 3*n_samples))
        if n_samples == 1:
            axes = [axes]
        
        for idx in range(n_samples):
            sample_data = df.iloc[idx][numeric_cols[:n_features]].values
            axes[idx].plot(sample_data, marker='o', markersize=3)
            axes[idx].set_title(f'Sample {idx + 1} - First {n_features} features')
            axes[idx].set_xlabel('Feature Index')
            axes[idx].set_ylabel('Value')
            axes[idx].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nNote: Showing first {n_features} features from first {n_samples} samples")
        print(f"Total features: {len(numeric_cols)}")

## Conclusion

This notebook has explored the structure and characteristics of the Alzheimer's EEG dataset. 

**Key Findings:**
- Dataset structure and file organization
- Data types and missing values
- Distribution of target variables (if present)
- Basic statistics and correlations
- Potential EEG channel information

**Next Steps (for ML):**
- Data preprocessing and cleaning
- Feature engineering
- Train/test split
- Model selection and training
- Evaluation metrics

---

*Notebook created for exploring the Largest Alzheimer EEG Dataset from Kaggle*